# Probability and Measurement (No Quantum Yet)

Quantum measurement is probabilistic. Before we make it quantum we will get very comfortable with classical probability — distributions, sampling, expectation values, and the law of large numbers.

**Objectives:**
- Define and visualize discrete probability distributions
- Sample from a distribution and watch empirical frequencies converge
- Compute expectation values
- Preview the Born rule: quantum measurement is **just** sampling from a   distribution computed from a state vector

**Reference:** See [`../GUIDE.md`](../GUIDE.md).

<!-- browser-runnable -->


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. A probability distribution is just a list of numbers

Two rules:

1. Every entry is `>= 0`.
2. They sum to `1`.

That is the whole definition.

In [ ]:
outcomes = ['heads', 'tails']
probs = np.array([0.7, 0.3])           # a biased coin

assert np.all(probs >= 0)
assert np.isclose(probs.sum(), 1.0)

plt.bar(outcomes, probs, color=['#ff9900', '#146eb4'])
plt.ylabel('Probability')
plt.title('Biased coin distribution')
plt.ylim(0, 1)
plt.show()

## 2. Sampling: turning a distribution into actual outcomes

If you flip the biased coin once, you might get either result. If you flip it many times and tally, the fraction of heads will get close to 0.7.

In [ ]:
def flip(n_shots, probs):
    return np.random.choice(len(probs), size=n_shots, p=probs)

for n in [10, 100, 1000, 100_000]:
    samples = flip(n, probs)
    p_heads = np.mean(samples == 0)
    print(f'{n:>7} shots  ->  empirical P(heads) = {p_heads:.4f}')

Empirical frequency converges to the true probability as the number of shots grows. This is the **law of large numbers**. In quantum computing, the "shots" argument you pass to a simulator or QPU is exactly this idea.

## 3. Expectation value

If each outcome `i` has a numerical value `v_i` and probability `p_i`, the **expectation value** is

$$\langle V \rangle = \sum_i p_i \, v_i$$

It is just a weighted average.

In [ ]:
# Suppose heads pays $1 and tails costs $0.50. Expected payoff per flip?
values = np.array([1.0, -0.5])
ev = (probs * values).sum()
print(f'Expected payoff = ${ev:.3f}')

# Confirm empirically with many flips
samples = flip(100_000, probs)
payoffs = values[samples]
print(f'Empirical average over 100k flips = ${payoffs.mean():.3f}')

## 4. Visualizing convergence

Watch the running average settle as samples accumulate.

In [ ]:
samples = flip(2000, probs)
running_heads_frac = np.cumsum(samples == 0) / np.arange(1, len(samples) + 1)

plt.plot(running_heads_frac, color='#ff9900', linewidth=1)
plt.axhline(0.7, color='gray', linestyle='--', label='true P(heads)')
plt.xlabel('Number of flips')
plt.ylabel('Empirical P(heads)')
plt.title('Empirical frequency converging to true probability')
plt.legend()
plt.show()

## 5. A first taste of the Born rule

Here is a tiny preview of where this is heading. In quantum mechanics, a state of a single qubit is a 2-vector

$$|\psi\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}, \qquad |\alpha|^2 + |\beta|^2 = 1.$$

When you **measure** it in the computational basis, you get outcome `0` with probability $|\alpha|^2$ and outcome `1` with probability $|\beta|^2$.

That is the entire "Born rule". Quantum measurement is sampling from the distribution `[|alpha|^2, |beta|^2]`.

In [ ]:
# Example: a single qubit in state (sqrt(0.7), sqrt(0.3))
psi = np.array([np.sqrt(0.7), np.sqrt(0.3)])
print('norm =', np.linalg.norm(psi))     # 1

probs_quantum = np.abs(psi) ** 2
print('measurement distribution:', probs_quantum)

# Simulate 'measuring' this qubit 1000 times
outcomes = np.random.choice([0, 1], size=1000, p=probs_quantum)
print('empirical fraction of 0s:', np.mean(outcomes == 0))

Notice that the quantum part is just one line: `np.abs(psi) ** 2`. Everything else is ordinary probability. We will build out exactly how `psi` is constructed in the next two notebooks.

## 6. Exercises

Three exercises to lock in the toolkit: sampling a distribution, expectation
values, and the Born rule. Each has tiered hints -- expand only what you
need -- and a check cell that tells you when you have it. Worked solutions
stay at the bottom of the notebook, but give the checks an honest attempt
first.

### Exercise 1 — A four-outcome distribution

Coins have two outcomes; nothing about probability stops at two. Build the
distribution `[0.1, 0.2, 0.3, 0.4]` over outcomes A, B, C, D, sample it
50,000 times, and see the law of large numbers hold for every outcome at once.

Define `dist4`: the four probabilities as an array, in the order above, and
`emp_freqs`: the empirical frequency of each outcome across your 50,000
samples, in the same order.

<details><summary>Hint 1 — nudge</summary>

Section 2 sampled a two-outcome distribution with `np.random.choice`; the
same call takes any number of outcomes. An empirical frequency is a count
divided by the total number of samples.

</details>
<details><summary>Hint 2 — approach</summary>

Draw integer outcomes with `np.random.choice(4, size=..., p=...)`, tally how
many times each of 0..3 appears (`np.bincount(..., minlength=4)` does it in
one call), then divide the tallies by 50,000.

</details>

In [ ]:
# Exercise 1: Sample a four-outcome distribution and check convergence.
# Define: dist4 -- the distribution [0.1, 0.2, 0.3, 0.4], and
# emp_freqs -- the empirical frequency of each outcome over 50,000 samples.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert len(dist4) == 4, "the distribution covers four outcomes"
    assert np.all(np.asarray(dist4) >= 0) and np.isclose(np.sum(dist4), 1.0), (
        "dist4 must follow the two rules from Section 1"
    )
    assert np.allclose(dist4, [0.1, 0.2, 0.3, 0.4]), (
        "use the four probabilities from the prompt, in order"
    )
    assert len(emp_freqs) == 4, "record one empirical frequency per outcome"
    assert np.all(np.asarray(emp_freqs) >= 0), "frequencies are never negative"
    assert np.isclose(np.sum(emp_freqs), 1.0), (
        "frequencies across all outcomes must sum to 1 -- divide counts by the total"
    )
    assert np.max(np.abs(np.asarray(emp_freqs) - np.asarray(dist4))) < 0.015, (
        "50,000 samples should put every empirical frequency within about 0.01 "
        "of the truth -- check your tally and normalization"
    )

### Exercise 2 — Expectation value of a die

A fair die has faces 1 through 6, all equally likely. Compute the expectation
value of the face two ways and watch them agree.

Define `die_ev`: the analytic expectation value from the Section 3 formula
(no sampling), and `die_ev_sampled`: the average face over at least 100,000
simulated rolls.

<details><summary>Hint 1 — nudge</summary>

The expectation value is a weighted average: each face times its probability,
summed. For a fair die all six weights are equal -- what must each one be for
the distribution rules from Section 1 to hold?

</details>
<details><summary>Hint 2 — approach</summary>

Build `np.arange(1, 7)` for the faces and `np.full(6, ...)` for their
probabilities, then multiply and sum for `die_ev`. For `die_ev_sampled`,
draw rolls with `np.random.choice` over the faces and take their `.mean()`.

</details>

In [ ]:
# Exercise 2: Expectation value of a fair die, analytically and by sampling.
# Define: die_ev -- the analytic expectation value, and
# die_ev_sampled -- the average of at least 100,000 simulated rolls.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert 1.0 <= die_ev <= 6.0, "an average of faces 1..6 must land between 1 and 6"
    assert np.isclose(die_ev, 3.5), (
        "recompute the weighted average: each face times its probability, summed"
    )
    assert 1.0 <= die_ev_sampled <= 6.0, (
        "a sampled average of faces 1..6 must land between 1 and 6"
    )
    assert abs(die_ev_sampled - die_ev) < 0.05, (
        "100,000 rolls should agree with the analytic value to within a few hundredths"
    )

### Exercise 3 — Born rule by hand

Apply Section 5 to a state you have not seen yet:

$$|\psi\rangle = \begin{pmatrix} 1/\sqrt{3} \\ \sqrt{2}/\sqrt{3} \end{pmatrix}$$

Define `psi_ex`: that state as a NumPy array, and `p_zero`, `p_one`: the
probabilities of measuring `0` and `1`.

<details><summary>Hint 1 — nudge</summary>

The Born rule was one line: measurement probabilities are squared amplitude
magnitudes. The first amplitude belongs to outcome `0`, the second to
outcome `1`.

</details>
<details><summary>Hint 2 — approach</summary>

Build the vector with `np.array` and `np.sqrt`, then apply the same
`np.abs(...) ** 2` move as the Section 5 code cell and read off the two
entries. Sanity-check that your two probabilities sum to 1 before running
the check.

</details>

In [ ]:
# Exercise 3: Apply the Born rule to psi = [1/sqrt(3), sqrt(2)/sqrt(3)].
# Define: psi_ex -- the state as a NumPy array, and
# p_zero, p_one -- the probabilities of measuring 0 and 1.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert len(psi_ex) == 2, "a single-qubit state is a 2-vector"
    assert np.isclose(np.linalg.norm(psi_ex), 1.0), (
        "a valid state vector has norm 1 -- check the amplitudes"
    )
    assert 0.0 <= p_zero <= 1.0 and 0.0 <= p_one <= 1.0, "probabilities live in [0, 1]"
    assert np.isclose(p_zero + p_one, 1.0), (
        "the two outcomes exhaust the possibilities, so their probabilities sum to 1"
    )
    assert np.isclose(p_one, 2 * p_zero), (
        "the second amplitude is sqrt(2) times the first -- "
        "what does squaring do to that ratio?"
    )

### Solutions

In [ ]:
# --- Exercise 1 ---
dist4 = np.array([0.1, 0.2, 0.3, 0.4])
samples4 = np.random.choice(4, size=50_000, p=dist4)
emp_freqs = np.bincount(samples4, minlength=4) / 50_000
print('true     :', dist4)
print('empirical:', emp_freqs)
print('all within 0.01?', np.all(np.abs(emp_freqs - dist4) < 0.01))

# --- Exercise 2 ---
faces = np.arange(1, 7)
face_probs = np.full(6, 1/6)
die_ev = (faces * face_probs).sum()
die_ev_sampled = np.random.choice(faces, size=100_000).mean()
print('EV (analytic) =', die_ev)                   # 3.5
print('EV (sampled)  =', die_ev_sampled)

# --- Exercise 3 ---
psi_ex = np.array([1/np.sqrt(3), np.sqrt(2)/np.sqrt(3)])
p_zero = abs(psi_ex[0])**2
p_one = abs(psi_ex[1])**2
print('P(0) =', p_zero)                # 1/3
print('P(1) =', p_one)                 # 2/3

**You finished notebook 3.** Move on to [`04-what-is-a-qubit.ipynb`](04-what-is-a-qubit.ipynb).